# Automatic false-positive cleanup

Run this **after** detection has produced `*_detections.csv` (e.g. `scripts/run_detection_ipa.py` writing to `outputs/detections/IPA1ST/`).

Three automatic filters — no manual listening required:

1. **Mahalanobis OOD** — distance of each detection's feature vector to its predicted class's training cluster. The merged `Cernic` label is scored against the nearest of its three call-type clusters.
2. **YAMNet cross-check** — Google's 521-class audio tagger flags windows whose top class is Bird / Insect / Wind / Rain / Speech, etc.
3. **Temporal isolation** — a detection with no same-species neighbour within +/-30 s is suspicious (primates call in bouts).

A detection is kept only if **all three** filters agree. Detections flagged by **>=2** filters are saved under `auto_flagged_fp/<reason>/` as hard negatives for the next retraining iteration.

The logic lives in `src/auto_cleanup.py`; this notebook just calls `auto_cleanup.run_auto_cleanup(...)`.

## Step 0 — GPU check

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs:       {gpus}')
if not gpus:
    print('\nWARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')

## Step 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone the repo

In [ ]:
%cd /content
!rm -rf primates-sound-detection
!git clone -q https://github.com/Mo119m/primates-sound-detection.git
%cd /content/primates-sound-detection
!git log --oneline -3

## Step 3 — Install deps (adds tensorflow-hub for YAMNet)

In [ ]:
!pip install -q -r requirements.txt
print('\nDone.')

## Step 4 — Run the auto-cleanup

All three filters now live in `src/auto_cleanup.py`, so this notebook is just a thin driver — the logic is reproducible and callable from anywhere via `auto_cleanup.run_auto_cleanup(...)`.

Point `DETECTION_DIR` at the folder holding your `*_detections.csv` (searched recursively, so a station folder like `detections/IPA1ST` works).

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, '/content/primates-sound-detection/src')
os.environ.setdefault('PRIMATE_DATA_ROOT', '/content/drive/MyDrive/primates-data')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config, auto_cleanup

DETECTION_DIR = '/content/drive/MyDrive/primates-data/outputs/detections/IPA1ST'

result = auto_cleanup.run_auto_cleanup(detection_dir=DETECTION_DIR)
det_df = result['det_df']
CLEANUP_DIR = Path(result['output_dir'])
result['summary']

## Step 5 — Visualise the verdict

Per-species clean/flagged breakdown and which filters agreed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) stacked bar: clean / 1-flag / 2-flags / 3-flags per species
order = [s for s in config.CLASS_NAMES if s in det_df['species'].unique()]
counts = pd.DataFrame({
    'clean':   [int(((det_df.species == s) & (det_df.n_flags == 0)).sum()) for s in order],
    '1 flag':  [int(((det_df.species == s) & (det_df.n_flags == 1)).sum()) for s in order],
    '2 flags': [int(((det_df.species == s) & (det_df.n_flags == 2)).sum()) for s in order],
    '3 flags': [int(((det_df.species == s) & (det_df.n_flags == 3)).sum()) for s in order],
}, index=order)
colors = ['#27AE60', '#F1C40F', '#E67E22', '#C0392B']
counts.plot(kind='bar', stacked=True, color=colors, ax=axes[0], edgecolor='black', linewidth=0.5)
axes[0].set_title('Auto-cleanup verdict per species')
axes[0].set_ylabel('Detections')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(fontsize=9)

# (b) flag overlap Venn-ish bar
venn_rows = {
    'Mahal only':   int(((det_df.flag_mahal) & (~det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
    'YAMNet only':  int(((~det_df.flag_mahal) & (det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
    'Isolated only':int(((~det_df.flag_mahal) & (~det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    'M+Y':          int(((det_df.flag_mahal) & (det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
    'M+I':          int(((det_df.flag_mahal) & (~det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    'Y+I':          int(((~det_df.flag_mahal) & (det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    'All 3':        int(((det_df.flag_mahal) & (det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
}
axes[1].bar(list(venn_rows.keys()), list(venn_rows.values()),
            color=['#F1C40F', '#F1C40F', '#F1C40F', '#E67E22', '#E67E22', '#E67E22', '#C0392B'],
            edgecolor='black', linewidth=0.6)
axes[1].set_title('Filter agreement')
axes[1].set_ylabel('Detections')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
out = CLEANUP_DIR / 'auto_cleanup_summary.png'
plt.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Summary chart saved to {out}')

## Done

Outputs are under `outputs/auto_cleanup/`:

- `clean_detections.csv` — passed all three filters; trust without listening
- `suspicious_detections.csv` — flagged, with a `flag_reason` column
- `auto_flagged_fp/<reason>/*.wav` — >=2-flag clips to fold into Background

**Next retraining tip:** copy everything under `auto_flagged_fp/` into a new background folder (e.g. `background/auto_hard_negatives/`), add it to `config.BACKGROUND_FOLDERS`, and retrain. Each iteration should shrink the false-positive set.

**Reproduce from the command line instead:**

```
python scripts/run_auto_cleanup.py --detection-dir <dir-with-CSVs>
```